In [ ]:
# ==========================================
# PHASE 1: ENVIRONMENT CONFIGURATION & PARSING
# ==========================================
!pip install -q pymupdf chromadb transformers reportlab gradio langchain-text-splitters torch

import os
import time
import fitz  # PyMuPDF
import chromadb
import torch
import gradio as gr
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils import embedding_functions
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Clean workspace directories to guarantee state reproducibility
!rm -rf hackathon_pdfs chroma_db
os.makedirs("hackathon_pdfs", exist_ok=True)

print("📝 Generating mock multi-page technical PDFs for verification...")
def create_mock_pdf(filename, title, content_list):
    path = os.path.join("hackathon_pdfs", filename)
    c = canvas.Canvas(path, pagesize=letter)
    for page_idx, page_content in enumerate(content_list):
        c.setFont("Helvetica-Bold", 16)
        c.drawString(50, 750, f"{title} - Chapter {page_idx + 1}")
        c.setFont("Helvetica", 11)
        textobject = c.beginText(50, 710)
        textobject.setLeading(14)
        for line in page_content.split('\n'):
            textobject.textLine(line)
        c.drawText(textobject)
        c.setFont("Helvetica-Oblique", 9)
        c.drawString(50, 40, f"Source: {filename} | Page {page_idx + 1}")
        c.showPage()
    c.save()

# Define grounded knowledge baseline corpora
doc1_content = [
    "Artificial Intelligence and Machine Learning form the backbone of modern automation.\nSupervised learning requires labeled training data consisting of inputs and expected outputs.\nUnsupervised learning explores hidden structures in unlabeled data collections.",
    "Neural Networks mimic biological neurons using layers of mathematical weights and biases.\nDeep learning architectures utilize multiple hidden layers to extract high-level features.\nGradient descent optimization is applied iteratively to minimize total training loss functions."
]
doc2_content = [
    "Retrieval-Augmented Generation bridges the gap between static LLMs and dynamic private data.\nRAG workflows query external vector indices to append relevant semantic context into prompts.\nThis foundational technique significantly minimizes artificial intelligence model hallucinations.",
    "Vector databases like ChromaDB use specialized indices to manage embedding lookups.\nHNSW indices construct high-dimensional graphical pathways for fast approximate nearest neighbor search.\nCosine similarity evaluates the angular direction between dense semantic feature arrays."
]
doc3_content = [
    "Technical writing requires extreme precision, concise sentence layout, and structural clarity.\nAlways declare definitions immediately before introducing advanced architectural abstractions.\nKeep operational summaries scannable by deploying clear bulleted outlines and tables.",
    "Software documentation should highlight clear installation steps, input vectors, and output formats.\nVersion control platforms like GitHub safeguard source code reproducibility across teams.\nComprehensive engineering README files improve cross-functional peer-review processes."
]

create_mock_pdf("ai_foundations.pdf", "AI Foundations Book", doc1_content)
create_mock_pdf("rag_architecture.pdf", "RAG Systems Guide", doc2_content)
create_mock_pdf("technical_writing.pdf", "Technical Style Manual", doc3_content)

# ==========================================
# PHASE 2: RECURSIVE CHUNKING & EMBEDDINGS VECTOR DB
# ==========================================
print("\n🔀 Text extraction and structural segmentation in progress...")
def extract_and_chunk_all_pdfs(folder_path, chunk_size=400, chunk_overlap=50):
    all_chunks = []
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len)
    for filename in os.listdir(folder_path):
        if filename.endswith('.pdf'):
            pdf_path = os.path.join(folder_path, filename)
            doc = fitz.open(pdf_path)
            for page_num in range(len(doc)):
                page = doc[page_num]
                text = page.get_text()
                page_chunks = text_splitter.split_text(text)
                for chunk in page_chunks:
                    all_chunks.append({"text": chunk, "metadata": {"source": filename, "page": page_num + 1}})
    return all_chunks

processed_chunks = extract_and_chunk_all_pdfs("hackathon_pdfs")

print("📦 Building localized vector index using ChromaDB (HNSW Space)...")
chroma_client = chromadb.PersistentClient(path="./chroma_db_final")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = chroma_client.get_or_create_collection(name="hackathon_rag_collection", embedding_function=embedding_fn, metadata={"hnsw:space": "cosine"})

documents, metadatas, ids = [], [], []
for idx, chunk in enumerate(processed_chunks):
    documents.append(chunk["text"])
    metadatas.append(chunk["metadata"])
    ids.append(f"id_{idx}")

collection.add(documents=documents, metadatas=metadatas, ids=ids)
print(f"✅ Successfully synchronized Vector DB! Indexed: {collection.count()} active structural chunks.")


In [ ]:
# ==========================================
# PHASE 3: CAUSAL LLM INFERENCE PIPELINE (QWEN)
# ==========================================
print("⏳ Spawning local open-source LLM model weights on T4 VRAM... (1-2 mins)")
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("🚀 Generative AI Inference pipeline safely anchored on GPU.")

# Core Routing logic for RAG query pipeline
def ask_rag_chatbot(query):
    start_time = time.time()

    # 1. High-speed vector similarity match
    search_results = collection.query(query_texts=[query], n_results=2)
    retrieved_docs = search_results['documents'][0]
    retrieved_metadatas = search_results['metadatas'][0]

    # 2. Build structured prompt grounding window
    context_text = ""
    citations = []
    for doc, meta in zip(retrieved_docs, retrieved_metadatas):
        context_text += f"\n[Source: {meta['source']}, Page: {meta['page']}]\nContent: {doc}\n"
        citation_str = f"📄 {meta['source']} (Page {meta['page']})"
        if citation_str not in citations:
            citations.append(citation_str)

    # 3. Apply strict instruction routing guardrails to avoid hallucinations
    messages = [
        {"role": "system", "content": "You are a precise, helpful AI assistant. Answer the user's question using ONLY the provided text context. If the answer cannot be found in the context, politely say that you cannot find the answer. Always be factual and direct."},
        {"role": "user", "content": f"Context information:\n{context_text}\n\nQuestion: {query}\nAnswer:"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = generator(prompt, max_new_tokens=150, do_sample=False)

    # Corrected indexing fix to resolve previous dictionary lookup error
    generated_text = outputs[0]['generated_text'][len(prompt):].strip()
    latency = f"{round(time.time() - start_time, 2)} seconds"
    sources_output = "\n".join(citations) if citations else "No explicitly referenced context matched."

    return generated_text, sources_output, latency

# ==========================================
# PHASE 4: INTERACTIVE DEMO SURFACE (GRADIO UI)
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Open-Source Grounded RAG Chatbot Engine")
    gr.Markdown("### Real-time Knowledge Base Query System with Traceable Source Citations")

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(label="Enter Query:", placeholder="e.g., What are the key instructions for technical writing?")
            submit_btn = gr.Button("Query Knowledge Base", variant="primary")
        with gr.Column(scale=1):
            latency_out = gr.Label(label="Measured Latency (Target: 2-5s)")

    with gr.Row():
        answer_out = gr.Textbox(label="Generated Answer (Context Grounded):", interactive=False, lines=5)
        sources_out = gr.Textbox(label="Document Provenance (Page Citations):", interactive=False, lines=3)

    submit_btn.click(
        fn=ask_rag_chatbot,
        inputs=[query_input],
        outputs=[answer_out, sources_out, latency_out]
    )

# Fire up live sharable pipeline link
demo.launch(share=True, debug=True)
